# AIGC Detector — Training on Colab

Runs `train.py` against the real dataset on a Colab GPU. CPU-only training locally is impractical (90k+ images, frozen CLIP forward pass every step).

**Before running:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU** (or better) → Save.

Checkpoints are written to Google Drive (not Colab's ephemeral local disk), so a disconnect doesn't lose progress — `train.py` auto-resumes from the latest checkpoint (`resume_from_latest: true` in `configs/train.yaml`) the next time this notebook runs.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — set Runtime > Change runtime type > GPU")

## 1. Mount Google Drive

Used for checkpoint persistence across session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set secrets

`train.py` only needs `HF_TOKEN` (for the CLIP backbone download — works without it, just rate-limited as an anonymous request). Add it via Colab's Secrets manager: left sidebar → key icon → **New secret** → name `HF_TOKEN`, paste a token from [huggingface.co](https://huggingface.co) (account settings → Access Tokens) → toggle **Notebook access** on.

(`KAGGLE_USERNAME`/`KAGGLE_KEY` and `WANDB_API_KEY` are NOT needed here — they're only used by `data/prepare_datasets.py`, which you don't need to run since the dataset is already committed in the repo.)

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set.")
except Exception as e:
    print("No HF_TOKEN secret configured — continuing without one (downloads will be rate-limited, not blocked).")

## 3. Clone the repo

Pulls whatever is currently pushed to GitHub — make sure your local commits are pushed before running this.

In [ ]:
!git clone https://github.com/windyheng/choochoo.git
%cd choochoo

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Point checkpoints at Drive

Symlinks `checkpoints/` (what `configs/train.yaml`'s `checkpoint_dir` points to) into Drive, so it survives a session reset.

In [ ]:
!mkdir -p /content/drive/MyDrive/choochoo_checkpoints
!ln -sfn /content/drive/MyDrive/choochoo_checkpoints checkpoints
!ls -la checkpoints

## 6. Train

Re-run this same cell after a disconnect/reconnect (remount Drive first) — it resumes automatically from the latest checkpoint.

In [ ]:
!python train.py --config configs/train.yaml

## (Optional) Train the ablation branches

`clip_only` and `artifact_only` are separately-trained models (different `FusionHead` shape each) — needed for `evaluate.py`'s branch-comparison ablation. Each needs its own `checkpoint_dir`; easiest is a separate Drive folder per branch.

In [ ]:
# !mkdir -p /content/drive/MyDrive/choochoo_checkpoints_clip_only
# !rm -f checkpoints && ln -sfn /content/drive/MyDrive/choochoo_checkpoints_clip_only checkpoints
# !python train.py --config configs/train.yaml --branch clip_only

# !mkdir -p /content/drive/MyDrive/choochoo_checkpoints_artifact_only
# !rm -f checkpoints && ln -sfn /content/drive/MyDrive/choochoo_checkpoints_artifact_only checkpoints
# !python train.py --config configs/train.yaml --branch artifact_only